In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =====================================
# LOAD DATA
# =====================================

nav = pd.read_csv("../data/raw/02_nav_history.csv")
transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")
portfolio = pd.read_csv("../data/raw/09_portfolio_holdings.csv")

nav['date'] = pd.to_datetime(nav['date'])
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])

print("Datasets loaded successfully")

# =====================================
# DAILY RETURNS
# =====================================

pivot_nav = nav.pivot_table(
    index='date',
    columns='amfi_code',
    values='nav'
)

returns = pivot_nav.pct_change().dropna()

# =====================================
# VaR AND CVaR
# =====================================

var_list = []

for fund in returns.columns:

    fund_returns = returns[fund].dropna()

    var95 = np.percentile(fund_returns,5)

    cvar95 = fund_returns[fund_returns <= var95].mean()

    var_list.append(
        [fund,var95,cvar95]
    )

var_df = pd.DataFrame(
    var_list,
    columns=['amfi_code','VaR_95','CVaR_95']
)

var_df.to_csv(
    "../reports/var_cvar_report.csv",
    index=False
)

print("VaR Report Saved")

# =====================================
# ROLLING SHARPE RATIO
# =====================================

top5 = performance.head(5)['amfi_code']

plt.figure(figsize=(12,6))

for fund in top5:

    if fund in returns.columns:

        rolling_sharpe = (
            returns[fund].rolling(90).mean()
            /
            returns[fund].rolling(90).std()
        ) * np.sqrt(252)

        plt.plot(
            rolling_sharpe,
            label=str(fund)
        )

plt.legend()

plt.title(
    "Rolling 90-Day Sharpe Ratio"
)

plt.tight_layout()

plt.savefig(
    "../reports/rolling_sharpe_chart.png"
)

plt.close()

print("Rolling Sharpe Chart Saved")

# =====================================
# INVESTOR COHORT ANALYSIS
# =====================================

transactions['cohort_year'] = (
    transactions
    .groupby('investor_id')
    ['transaction_date']
    .transform('min')
    .dt.year
)

cohort = (
    transactions
    .groupby('cohort_year')
    ['amount_inr']
    .agg(['mean','sum'])
)

print("\nInvestor Cohort Analysis")
print(cohort)

# =====================================
# SIP CONTINUITY ANALYSIS
# =====================================

sip_txn = transactions[
    transactions['transaction_type']=="SIP"
]

gap_analysis = (
    sip_txn
    .sort_values(
        ['investor_id','transaction_date']
    )
)

gap_analysis['gap_days'] = (
    gap_analysis
    .groupby('investor_id')
    ['transaction_date']
    .diff()
    .dt.days
)

risk_gap = (
    gap_analysis
    .groupby('investor_id')
    ['gap_days']
    .mean()
)

at_risk = risk_gap[
    risk_gap > 35
]

print("\nAt Risk Investors")
print(at_risk.head())

# =====================================
# HHI CONCENTRATION
# =====================================

hhi = (
    portfolio
    .groupby('amfi_code')['weight_pct']
    .apply(
        lambda x:
        ((x/100)**2).sum()
    )
)

print("\nPortfolio HHI")
print(hhi.head())

print("\nAdvanced Analytics Completed")

Datasets loaded successfully
VaR Report Saved
Rolling Sharpe Chart Saved

Investor Cohort Analysis
                      mean         sum
cohort_year                           
2024         107422.541832  3491125187
2025         109158.577061    30455243

At Risk Investors
investor_id
INV000001     76.00
INV000002    207.00
INV000003    238.00
INV000004     85.40
INV000006    123.75
Name: gap_days, dtype: float64

Portfolio HHI
amfi_code
100016    0.139534
100033    0.147592
101206    0.129332
101207    0.200700
102885    0.174709
Name: weight_pct, dtype: float64

Advanced Analytics Completed
